`conda activate r_env`

Idea taken from https://doi.org/10.1038/s41588-022-01243-4 <br>
Code obtained from https://github.com/elo073/5loclung/blob/main/Explained%20Variability/

In [ ]:
suppressPackageStartupMessages({
    library(Seurat)
    library(dplyr)
    library(here)
    library(tibble)
    library(dittoSeq)
    library(stringr)
    library(data.table)
    library(tidyverse)
    library(patchwork) 
    library(RColorBrewer) 
    library(reticulate)
    options(stringsAsFactors = FALSE)
    set.seed(123)
})


source(here("scripts/style.R")) 
source(here("scripts/Barplot.LFLMM.R")) 
source(here("scripts/LFLMM.LMMBF.R")) 
source(here("scripts/functions.R")) 

ggplot2::theme_set(theme_min())

In [ ]:
dmg <- readRDS('/projects/0/einf2548/cruiz/dmg/data/rna_dmg_atlas_scglue_embbeding.rds')
dmg

An object of class Seurat 
19248 features across 397794 samples within 1 assay 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 2 dimensional reductions calculated: pca, umap

In [ ]:
dmg <- AddMetaData(dmg, readRDS('../../data/dmg_atlas_final_annotation.rds'))

### All cells

In [ ]:
# Define the percentage of cells you want to subset (e.g., 30%)
subset_percentage <- 0.3

# Get the number of cells to subset
num_cells_subset <- round(ncol(dmg) * subset_percentage)

# Randomly sample cell indices
subset_indices <- sample(1:ncol(dmg), num_cells_subset, replace = FALSE)

# Use the subset function to subset the Seurat object
dmg_subset <- subset(dmg, cells = subset_indices)
dmg_subset

An object of class Seurat 
19248 features across 119338 samples within 1 assay 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 2 dimensional reductions calculated: pca, umap

In [ ]:
norm_counts <- as.data.frame(GetAssayData(dmg_subset, slot='data'))

In [ ]:
dim(norm_counts)
percent <- dim(norm_counts)[2]*0.95
norm_counts <- norm_counts[rowSums(norm_counts == 0) <= percent, ]
dim(norm_counts)

[1]  19248 119338

[1]  10509 119338

In [ ]:
mdata <- dmg_subset@meta.data %>% select(c('nCount_RNA', 'nFeature_RNA',
                                           'SampleID', 'percent_mito', 'percent_ribo', 
                                           'Batch_for_correction','Study', 'Location',
                                           'Tumor_subtype', 'lvl_1'))

In [ ]:
# Logaritmize Total counts and Number of Genes
mdata$nCount_RNA=log(mdata$nCount_RNA)
mdata$nFeature_RNA=log(mdata$nFeature_RNA)

In [ ]:
# fitting lmm
res = LFLMM(norm_counts, mdata)

           Intercept           nCount_RNA         nFeature_RNA 
                   1                    1                    1 
            SampleID         percent_mito         percent_ribo 
                  83                    1                    1 
Batch_for_correction                Study             Location 
                   4                    2                    6 
       Tumor_subtype                lvl_1 
                   4                    5 
[1] "matrix prep 2"
[1] "Y"
[1]  10509 119338
      BT042_PD_CCTGCATAGGTCACAG-1 BT042_PD_ACAAGCTGTGGCTAGA-1
A1BG                            0                           0
A2M                             0                           0
AAAS                            0                           0
AACS                            0                           0
AADAT                           0                           0
      BT042_PD_TACTTACCAAGTACCT-1 BT042_PD_TGCGACGCAGCTGTGC-1
A1BG                            0                 

In [ ]:
Barplot(res, "figures/explained_variance_all.pdf", "All cells")

,varexp,varse
lvl_1,7.3548646,7.4478374
SampleID,3.8517748,3.8656508
nCount_RNA,3.6522403,3.7482471
nFeature_RNA,3.0353326,3.1157683
percent_ribo,2.2785932,2.3389667
Batch_for_correction,0.3109651,0.3250645
percent_mito,0.1857722,0.1908358
Location,0.1644728,0.1722348
Study,0.1212621,0.1364270
Tumor_subtype,0.1040221,0.1099718


### Nuclei

In [ ]:
dmg_subset$Material <- recode(dmg_subset$Batch_for_correction,
                          '10Xv1_nuclei_multiome' = 'nuclei',
                           '10Xv3_cell_rna'='cell',
                           '10Xv3_nuclei_rna'='nuclei',
                           '10Xv3.1_nuclei_rna'='nuclei')

In [ ]:
# Use the subset function to subset the Seurat object
nuclei <- subset(dmg_subset, Material=='nuclei')
nuclei

An object of class Seurat 
19248 features across 102100 samples within 1 assay 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 2 dimensional reductions calculated: pca, umap

In [ ]:
norm_counts <- as.data.frame(GetAssayData(nuclei, slot='data'))

In [ ]:
dim(norm_counts)
percent <- dim(norm_counts)[2]*0.95
norm_counts <- norm_counts[rowSums(norm_counts == 0) <= percent, ]
dim(norm_counts)

[1]  19248 102100

[1]  10427 102100

In [ ]:
mdata <- nuclei@meta.data %>% select(c('nCount_RNA', 'nFeature_RNA',
                                           'SampleID', 'percent_mito', 'percent_ribo', 
                                           'Batch_for_correction','Study', 'Location',
                                           'Tumor_subtype', 'lvl_1'))
mdata

In [ ]:
# Logaritmize Total counts and Number of Genes
mdata$nCount_RNA=log(mdata$nCount_RNA)
mdata$nFeature_RNA=log(mdata$nFeature_RNA)
mdata

In [ ]:
# fitting lmm
res = LFLMM(norm_counts, mdata)

           Intercept           nCount_RNA         nFeature_RNA 
                   1                    1                    1 
            SampleID         percent_mito         percent_ribo 
                  78                    1                    1 
Batch_for_correction                Study             Location 
                   3                    2                    6 
       Tumor_subtype                lvl_1 
                   4                    5 
[1] "matrix prep 2"
[1] "Y"
[1]  10427 102100
      BT042_PD_CCTGCATAGGTCACAG-1 BT042_PD_ACAAGCTGTGGCTAGA-1
A1BG                            0                           0
A2M                             0                           0
AAAS                            0                           0
AACS                            0                           0
AADAT                           0                           0
      BT042_PD_TACTTACCAAGTACCT-1 BT042_PD_TGCGACGCAGCTGTGC-1
A1BG                            0                 

In [ ]:
Barplot(res, "figures/explained_variance_nuclei.pdf", "Nuclei")

,varexp,varse
lvl_1,7.70272382,7.7999562
SampleID,3.83271280,3.8467042
nCount_RNA,3.46183361,3.5534770
nFeature_RNA,2.77333305,2.8474250
percent_ribo,0.89173339,0.9158086
Batch_for_correction,0.23601060,0.2511576
Location,0.21470585,0.2239902
percent_mito,0.15064689,0.1547774
Tumor_subtype,0.11415087,0.1204881
Study,0.08663981,0.1020212


### Cell

In [ ]:
# Use the subset function to subset the Seurat object
cell <- subset(dmg_subset, Material=='cell')
cell

An object of class Seurat 
19248 features across 17238 samples within 1 assay 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 2 dimensional reductions calculated: pca, umap

In [ ]:
norm_counts <- as.data.frame(GetAssayData(cell, slot='data'))

In [ ]:
dim(norm_counts)
percent <- dim(norm_counts)[2]*0.95
norm_counts <- norm_counts[rowSums(norm_counts == 0) <= percent, ]
dim(norm_counts)

[1] 19248 17238

[1] 10142 17238

In [ ]:
mdata <- cell@meta.data %>% select(c('nCount_RNA', 'nFeature_RNA',
                                           'SampleID', 'percent_mito', 'percent_ribo', 
                                            'Location',
                                           'Tumor_subtype', 'lvl_1'))
mdata

In [ ]:
# Logaritmize Total counts and Number of Genes
mdata$nCount_RNA=log(mdata$nCount_RNA)
mdata$nFeature_RNA=log(mdata$nFeature_RNA)
mdata

In [ ]:
# fitting lmm
res = LFLMM(norm_counts, mdata)

    Intercept    nCount_RNA  nFeature_RNA      SampleID  percent_mito 
            1             1             1             5             1 
 percent_ribo      Location Tumor_subtype         lvl_1 
            1             3             2             5 
[1] "matrix prep 2"
[1] "Y"
[1] 10142 17238
      rna_P-6117_S-8370_GCTGCAGCACCATCCT-1 rna_P-6117_S-8370_GCAGCCACAGTGACAG-1
A1BG                             0.1316070                            0.1512442
A2M                              0.0000000                            0.0000000
AAAS                             0.1316070                            0.1512442
AACS                             0.2478906                            0.1512442
AAGAB                            0.2478906                            0.1512442
      rna_P-6117_S-8370_GACCAATTCCCTAATT-1 rna_P-6117_S-8370_GTGCGGTTCTCTAGGA-1
A1BG                             0.0000000                            0.4777703
A2M                              0.7656643                  

In [ ]:
Barplot(res, "figures/explained_variance_cells.pdf", "Cells")

,varexp,varse
lvl_1,7.4028200752,7.51351888
nFeature_RNA,5.9340976808,6.09616509
nCount_RNA,5.3998792217,5.54837044
SampleID,3.8439573558,3.91872568
percent_ribo,3.2615103945,3.34921455
percent_mito,0.1786556410,0.18381869
Location,0.0091202678,0.07468197
Tumor_subtype,0.0002942983,0.07699646
